In [8]:
# Bring in the two tools we need
import sqlite3     # the mini database engine into Python
import pandas as pd   # for loading CSVs and holding data in tables

# Load each year's CSV into a table ("DataFrame") in memory
df_2023 = pd.read_csv("2023_Delta_Delays.csv")
df_2024 = pd.read_csv("2024_Delta_Delays.csv")
df_2025 = pd.read_csv("2025_Delta_Delays.csv")

# Stack all three years into one combined table, *Double check what that exactly the pd.concat function does and why the second range is set as ignore_index=True**
df_all = pd.concat([df_2023, df_2024, df_2025], ignore_index=True)

# Create a temporary in-memory database and load the data into it as a table called "delays", ** EXPLAIN WHAT df_all.to_sql... is and does***
conn = sqlite3.connect(":memory:")
df_all.to_sql("delays", conn, if_exists="replace", index=False)

print("Data loaded successfully!")
print(f"Total rows: {len(df_all)}")
df_all.head()     # show the first 5 rows so you can see the columns 

Data loaded successfully!
Total rows: 5131


,year,month,carrier,carrier_name,airport,airport_name,arr_flights,arr_del15,carrier_ct,weather_ct,...,security_ct,late_aircraft_ct,arr_cancelled,arr_diverted,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,2023,12,DL,Delta Air Lines Network,IAH,"Houston, TX: George Bush Intercontinental/Houston",616.0,61.0,33.69,1.00,...,0.0,14.88,0.0,0.0,3794.0,2182.0,188.0,478.0,0.0,946.0
1,2023,12,DL,Delta Air Lines Network,ICT,"Wichita, KS: Wichita Dwight D Eisenhower National",67.0,3.0,2.00,0.00,...,0.0,0.00,0.0,1.0,165.0,146.0,0.0,19.0,0.0,0.0
2,2023,12,DL,Delta Air Lines Network,ILM,"Wilmington, NC: Wilmington International",73.0,5.0,5.00,0.00,...,0.0,0.00,0.0,0.0,203.0,203.0,0.0,0.0,0.0,0.0
3,2023,12,DL,Delta Air Lines Network,IND,"Indianapolis, IN: Indianapolis International",370.0,26.0,14.52,0.95,...,0.0,1.23,0.0,0.0,840.0,515.0,40.0,238.0,0.0,47.0
4,2023,12,DL,Delta Air Lines Network,JAC,"Jackson, WY: Jackson Hole",113.0,10.0,4.91,2.00,...,0.0,2.84,0.0,1.0,1232.0,158.0,956.0,12.0,0.0,106.0


In [29]:
# which airports have the worst delays? For each airport, add up total flights
# and delayed flights, work out what percentage, sort worst-first, show the top 15.
query1 = """
SELECT
    airport,
    airport_name,
    SUM(arr_flights) AS total_flights,
    SUM(arr_del15) AS total_delayed,
    ROUND(SUM(arr_del15) * 100.0 / SUM(arr_flights), 2) AS delay_rate_pct
FROM delays
GROUP BY airport, airport_name
ORDER BY delay_rate_pct DESC
LIMIT 15;

"""


result1 = pd.read_sql_query(query1, conn)
result1

,airport,airport_name,total_flights,total_delayed,delay_rate_pct
0,RAP,"Rapid City, SD: Rapid City Regional",203.0,63.0,31.03
1,FAT,"Fresno, CA: Fresno Yosemite International",422.0,127.0,30.09
2,SBN,"South Bend, IN: South Bend International",7.0,2.0,28.57
3,IDA,"Idaho Falls, ID: Idaho Falls Regional",450.0,118.0,26.22
4,EWR,"Newark, NJ: Newark Liberty International",19519.0,5074.0,26.00
5,MIA,"Miami, FL: Miami International",32862.0,8376.0,25.49
6,FSD,"Sioux Falls, SD: Joe Foss Field",2128.0,529.0,24.86
7,PBI,"West Palm Beach/Palm Beach, FL: Palm Beach Int...",20940.0,5163.0,24.66
8,DLH,"Duluth, MN: Duluth International",220.0,53.0,24.09
9,ALB,"Albany, NY: Albany International",3852.0,922.0,23.94


In [24]:
query1a = """
SELECT *
FROM (
    SELECT
        airport,
        airport_name,
        SUM(arr_flights) AS total_flights,
        SUM(arr_del15) AS total_delayed,
        ROUND(SUM(arr_del15) * 100.0 / SUM(arr_flights), 2) AS delay_rate_pct
    FROM delays
    GROUP BY airport, airport_name
    ORDER BY delay_rate_pct DESC
    LIMIT 15
)
ORDER BY total_delayed DESC;

"""


result1a = pd.read_sql_query(query1a, conn)
result1a

,airport,airport_name,total_flights,total_delayed,delay_rate_pct
0,MCO,"Orlando, FL: Orlando International",63355.0,14857.0,23.45
1,SFO,"San Francisco, CA: San Francisco International",39564.0,9274.0,23.44
2,MIA,"Miami, FL: Miami International",32862.0,8376.0,25.49
3,DFW,"Dallas/Fort Worth, TX: Dallas/Fort Worth Inter...",35334.0,8213.0,23.24
4,ORD,"Chicago, IL: Chicago O'Hare International",33637.0,7868.0,23.39
5,PBI,"West Palm Beach/Palm Beach, FL: Palm Beach Int...",20940.0,5163.0,24.66
6,EWR,"Newark, NJ: Newark Liberty International",19519.0,5074.0,26.00
7,ALB,"Albany, NY: Albany International",3852.0,922.0,23.94
8,FSD,"Sioux Falls, SD: Joe Foss Field",2128.0,529.0,24.86
9,BUR,"Burbank, CA: Bob Hope",1306.0,303.0,23.20


In [20]:
# What's causing the delays? Add up every delay by its cause 
# across all airports and years, so you can see which cause is biggest

query2 = """
SELECT
    ROUND(SUM(late_aircraft_ct), 0) AS late_aircraft,
    ROUND(SUM(carrier_ct), 0) AS carrier,
    ROUND(SUM(nas_ct), 0) AS air_traffic,
    ROUND(SUM(weather_ct), 0) AS weather
FROM delays;
"""

result2 = pd.read_sql_query(query2, conn)
result2

,late_aircraft,carrier,air_traffic,weather
0,159202.0,205425.0,151280.0,15567.0


In [27]:
# when during the year are delays worst?
# group every row by its month number and get the delay rate for each, Jan - Dec

query3 = """
SELECT
    month,
    SUM(arr_flights) AS total_flights,
    SUM(arr_del15) AS total_delayed,
    ROUND(SUM(arr_del15) * 100.0 / SUM(arr_flights), 2) AS delay_rate_pct
FROM delays
GROUP BY month
ORDER BY month DESC;

"""

result3 = pd.read_sql_query(query3, conn)
result3

,month,total_flights,total_delayed,delay_rate_pct
0,12,249444.0,45237.0,18.14
1,11,246653.0,30694.0,12.44
2,10,263764.0,30693.0,11.64
3,9,251121.0,32523.0,12.95
4,8,269767.0,48809.0,18.09
5,7,276113.0,69584.0,25.20
6,6,266116.0,60557.0,22.76
7,5,262771.0,49103.0,18.69
8,4,247230.0,42016.0,16.99
9,3,249946.0,46202.0,18.48


In [30]:
result1.to_csv("outputs/query1_airport_delays.csv", index=False)
result1a.to_csv("outputs/query1a_airport_delays.csv", index=False)
result2.to_csv("outputs/query2_delay_causes.csv", index=False)
result3.to_csv("outputs/query3_delay_causes.csv", index=False)
print("Results exported to the outputs/ folder.")

Results exported to the outputs/ folder.
